# TP — Réutiliser un modèle pré-entraîné avec Hugging Face

**Objectif.** Ce TP illustre trois façons de plus en plus impliquées de réutiliser un modèle
pré-entraîné :

1. l'utiliser **tel quel**, sans aucun entraînement (`pipeline`) ;
2. regarder **ce qu'il y a sous le capot** (tokenizer + modèle) ;
3. le réutiliser comme **extracteur de features gelé**, puis **dégeler** progressivement
   quelques couches pour l'adapter à une nouvelle tâche (analyse de sentiment sur des
   critiques de films en français, dataset *Allociné*).

On utilise la boîte à outils `training_toolbox.py` (à placer dans le même dossier que ce
notebook) pour ne pas ré-écrire la boucle d'entraînement : c'est la même API `Trainer` que
dans les TPs précédents sur les CNN et RNN.

**Prérequis.** `pip install torch transformers datasets` (décommentez la cellule suivante
sur Colab).

In [ ]:
# !pip install -q torch transformers datasets

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, AutoModel, pipeline
from datasets import load_dataset

from training_toolbox import Trainer, freeze, unfreeze, count_trainable_parameters, accuracy

torch.manual_seed(0)


## Partie 1 — Un modèle pré-entraîné, sans entraînement

Le plus simple pour réutiliser un modèle : le `pipeline` de `transformers`. Ici un modèle
déjà fine-tuné pour l'analyse de sentiment (multilingue, dont le français) — on ne fait
**aucun** entraînement, juste de l'inférence.

In [ ]:
sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)

phrases = [
    "Ce film est une merveille, j'ai adoré chaque minute.",
    "Un scénario sans intérêt et des acteurs fades.",
    "Correct sans plus, on a vu bien mieux dans le genre.",
]

for p in phrases:
    print(p, "->", sentiment(p))


**Questions.**

- Ce modèle renvoie une note de 1 à 5 étoiles plutôt qu'un simple positif/négatif : que
  se passe-t-il sur la phrase la plus neutre ?
- Sur quel type de texte pensez-vous que ce modèle a été entraîné ? Que se passe-t-il si
  vous testez une phrase très familière ou très technique ?

## Partie 2 — Sous le capot : tokenizer + modèle

Un `pipeline` masque deux objets : un **tokenizer** (texte -> identifiants numériques) et
un **modèle** (identifiants -> logits). Regardons-les séparément, avec un modèle de base
non spécialisé pour une tâche (`camembert-base`), qu'on va justement réutiliser en Partie 3.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("camembert-base")
backbone = AutoModel.from_pretrained("camembert-base")

encoded = tokenizer("Ce film est une merveille.", return_tensors="pt")
print(encoded)
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]))

with torch.no_grad():
    output = backbone(**encoded)

# output.last_hidden_state : un vecteur par token. On récupère celui du token spécial
# [CLS] (premier token), utilisé classiquement comme représentation de la phrase entière.
sentence_embedding = output.last_hidden_state[:, 0, :]
print(sentence_embedding.shape)  # (1, 768)


`camembert-base` sait produire une représentation vectorielle d'une phrase, mais ne sait
pas encore classer un sentiment : il n'a pas de tête de classification. C'est exactement la
même logique que le transfer learning vu sur les CNN (backbone pré-entraîné + nouvelle tête),
appliquée ici au texte.

## Partie 3 — Backbone gelé + tête entraînée

On charge un petit sous-ensemble du dataset *Allociné* (critiques de films en français,
sentiment binaire) pour que l'entraînement reste rapide sur CPU.

In [ ]:
raw = load_dataset("allocine")
train_small = raw["train"].shuffle(seed=0).select(range(3000))
val_small = raw["validation"].shuffle(seed=0).select(range(500))

def tokenize(batch):
    return tokenizer(batch["review"], truncation=True, padding="max_length", max_length=128)

train_small = train_small.map(tokenize, batched=True)
val_small = val_small.map(tokenize, batched=True)

columns = ["input_ids", "attention_mask", "label"]
train_small.set_format(type="torch", columns=columns)
val_small.set_format(type="torch", columns=columns)

def collate(batch):
    out = {k: torch.stack([b[k] for b in batch]) for k in ["input_ids", "attention_mask"]}
    out["labels"] = torch.stack([b["label"] for b in batch])
    return out

train_loader = DataLoader(train_small, batch_size=16, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_small, batch_size=32, collate_fn=collate)


In [ ]:
class CamembertClassifier(nn.Module):
    """Backbone pré-entraîné + une tête linéaire pour la classification binaire."""

    def __init__(self, backbone, n_classes=2):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(backbone.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = out.last_hidden_state[:, 0, :]
        return self.head(cls_embedding)


model = CamembertClassifier(backbone)

# On gèle tout le backbone : seule la tête (quelques centaines de paramètres) est entraînée.
freeze(model.backbone)
print("Paramètres entraînables :", count_trainable_parameters(model))

optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

trainer = Trainer(model, optimizer, loss_fn, metrics={"acc": accuracy})
history_frozen = trainer.fit(train_loader, val_loader, epochs=3)


## Partie 4 — Dégeler quelques couches

Le backbone gelé sert de simple extracteur de features génériques. En dégelant les
dernières couches du transformer (celles qui encodent les notions les plus spécifiques),
on peut souvent gagner en performance, au prix d'un entraînement plus coûteux.

In [ ]:
# On ne dégèle que la dernière couche du transformer (camembert-base en a 12, indices 0-11).
unfreeze(model.backbone.encoder.layer[-1])
print("Paramètres entraînables après dégel partiel :", count_trainable_parameters(model))

# Learning rate plus faible : on ne veut pas détruire ce que le backbone a appris.
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad], lr=2e-5
)

trainer = Trainer(model, optimizer, loss_fn, metrics={"acc": accuracy})
history_finetuned = trainer.fit(train_loader, val_loader, epochs=2)


**Question.** Comparez `history_frozen` et `history_finetuned` (accuracy de validation).
Le gain observé justifie-t-il le coût de calcul supplémentaire ? Que se passerait-il si on
dégelait *tout* le backbone d'un coup avec un jeu d'entraînement aussi petit (3000 exemples) ?

## Partie 5 (bonus) — Visualiser l'attention

On peut demander au modèle de renvoyer ses poids d'attention (`output_attentions=True`) pour
observer, pour une phrase donnée, quels tokens le modèle relie entre eux.

In [ ]:
import matplotlib.pyplot as plt

sentence = "Le film est long mais vraiment magnifique."
enc = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    out = model.backbone(**enc, output_attentions=True)

# Dernière couche, tête d'attention n°0
attn = out.attentions[-1][0, 0].numpy()
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(attn, cmap="viridis")
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=90)
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
ax.set_title("Attention (dernière couche, tête 0)")
plt.tight_layout()
plt.show()


## Pour aller plus loin (optionnel)

- Essayez un backbone plus petit (`distilcamembert-base`) : même démarche, entraînement
  plus rapide — utile pour un TP sur machine peu puissante.
- Remplacez la tête linéaire par un petit MLP avec dropout (lien avec le chapitre
  *Régularisation*).
- Utilisez `EarlyStopping` et `ModelCheckpoint` de `training_toolbox.py` pour arrêter
  l'entraînement automatiquement.
- Comparez ce transfer learning texte à celui vu sur les CNN (chapitre *Convolutional
  Neural Networks*) : quels points communs, quelles différences ?